# Sentinel-3 OLCI L1 EFR — direct GeoZarr access + deck.gl display

TiTiler visualization for the `sentinel-3-olci-l1-efr-staging` collection is
**gated off** pending converter-side fixes (multiscale group alignment + CRS
declaration — see `claude-docs/specs/s3_olci_pipeline_runbook.md`). This notebook
shows the path that **does** work today: read the converted multiscale GeoZarr
directly over public https and render it with [deck.gl](https://deck.gl)
(via `pydeck`).

OLCI L1 EFR is **swath data** — there is no native CRS or regular grid. The store
instead carries per-pixel `latitude`/`longitude` arrays, so we render the scene as
a coloured point cloud, one point per (subsampled) pixel.

**Requirements** (on top of the repo venv): `pip install pydeck`
(`zarr`, `numpy`, `pandas`, `requests` ship with the pipeline dependencies).

**Access note:** the staging bucket is public **GET-only** — objects can be fetched
over https, but S3 listing is not allowed. That is why we open arrays by explicit
path (taken from the STAC item's assets) instead of `xr.open_zarr` on the group,
which needs to enumerate children.

## 1. Find the item in the STAC catalogue

Every converted scene is registered in the STAC API. The `*_radianceData` assets
point at the individual band arrays; the `radianceData` asset points at the
`measurements` group that contains them all (plus `latitude`/`longitude`).

In [ ]:
import requests

STAC_API = "https://api.explorer.eopf.copernicus.eu/stac"
COLLECTION = "sentinel-3-olci-l1-efr-staging"

# The scene this notebook narrates (west-Greenland coast, 2026-07-21). If it has
# been retired, fall back to the newest item — everything below still works, only
# the Greenland-specific prose stops matching.
DEMO_ITEM = (
    "S3A_OL_1_EFR____20260721T142227_20260721T142527_20260721T162757_"
    "0179_142_039_1620_PS1_O_NR_004"
)

resp = requests.get(f"{STAC_API}/collections/{COLLECTION}/items/{DEMO_ITEM}", timeout=30)
if resp.status_code == 404:
    resp = requests.get(f"{STAC_API}/collections/{COLLECTION}/items", timeout=30)
    resp.raise_for_status()
    item = resp.json()["features"][0]
else:
    resp.raise_for_status()
    item = resp.json()

# The measurements group href; band arrays live directly underneath it.
measurements_href = item["assets"]["radianceData"]["href"]
print("item :", item["id"])
print("bbox :", item["bbox"])
print("group:", measurements_href)

## 2. Load an overview level

The converter writes a multiscale layout: full resolution (~300 m GSD, 4090×4865
for this scene) plus `r2`/`r4`/`r8` overviews (2× averaging each). For an
interactive overview map, `r8` (511×608 ≈ 310 k pixels, ~2.4 km/px) is plenty and
loads in a couple of seconds. `r4` or `r2` give more detail with proportionally
more points. Full resolution (`LEVEL = ""`) is ~19.9 M pixels — far too many for
the point-table rendering below; load a chunk slice instead (see Notes).

False-colour "natural" composite from the 21 `Oa*` radiance bands:
**Oa08** (665 nm) → red, **Oa06** (560 nm) → green, **Oa04** (490 nm) → blue.

In [ ]:
import numpy as np
import zarr

LEVEL = "r8"  # one of: "" (full res), "r2", "r4", "r8"


def load(name: str) -> np.ndarray:
    path = f"{measurements_href}/{LEVEL}/{name}" if LEVEL else f"{measurements_href}/{name}"
    return zarr.open_array(path, mode="r")[:]


red = load("oa08_radiance")
green = load("oa06_radiance")
blue = load("oa04_radiance")
lat = load("latitude")
lon = load("longitude")

valid = ~(np.isnan(red) | np.isnan(green) | np.isnan(blue) | np.isnan(lat) | np.isnan(lon))
print(f"grid {red.shape}, {int(valid.sum()):,} valid px")
print(
    f"lat {np.nanmin(lat):.2f}..{np.nanmax(lat):.2f}, lon {np.nanmin(lon):.2f}..{np.nanmax(lon):.2f}"
)

## 3. Build the RGB point table

Radiances are TOA W·m⁻²·sr⁻¹·µm⁻¹ floats — stretch each band between its 2nd and
98th percentile to 0–255 for display.

In [ ]:
import pandas as pd


def stretch(band: np.ndarray, lo: float = 2, hi: float = 98) -> np.ndarray:
    p_lo, p_hi = np.percentile(band[valid], [lo, hi])
    scaled = np.clip((band - p_lo) / (p_hi - p_lo) * 255, 0, 255)
    return np.nan_to_num(scaled).astype(np.uint8)


# Round coords to 4 dp (~11 m, far below the pixel footprint) — deck.gl serializes
# this table to JSON, so full float64 precision would bloat the export several-fold.
df = pd.DataFrame(
    {
        "lon": np.round(lon[valid], 4),
        "lat": np.round(lat[valid], 4),
        "r": stretch(red)[valid],
        "g": stretch(green)[valid],
        "b": stretch(blue)[valid],
    }
)
df["color"] = df[["r", "g", "b"]].values.tolist()
print(f"{len(df):,} points")
df.head(3)

## 4. Render with deck.gl

One `ScatterplotLayer` point per pixel, radius ≈ 0.6× the pixel footprint at the
chosen level so neighbouring points tile into a continuous image. Pan/zoom to
explore — this July scene runs down the west coast of Greenland.

In [ ]:
import pydeck as pdk

# Point radius scaled to the pixel footprint of the chosen level (base GSD 300 m).
SCALE = {"": 1, "r2": 2, "r4": 4, "r8": 8}[LEVEL]
POINT_RADIUS_M = 300 * SCALE * 0.6

layer = pdk.Layer(
    "ScatterplotLayer",
    df[["lon", "lat", "color"]],
    get_position=["lon", "lat"],
    get_fill_color="color",
    get_radius=POINT_RADIUS_M,
    pickable=True,
)

view = pdk.ViewState(
    longitude=float(df.lon.mean()),
    latitude=float(df.lat.mean()),
    zoom=4,
)

deck = pdk.Deck(
    layers=[layer],
    initial_view_state=view,
    map_provider="carto",
    map_style=pdk.map_styles.CARTO_LIGHT,
    tooltip={"text": "lon {lon}\nlat {lat}"},
)
# Renders inline. To keep a standalone page, pass a path — it's ~45 MB at r8
# (the whole point table is embedded), so don't drop it inside the repo.
deck.to_html()

## Notes

- **Full resolution / other bands:** set `LEVEL = ""` and pick any of
  `oa01_radiance` … `oa21_radiance` (wavelengths are in the collection's `bands`
  summary). Full-res arrays are 4090×4865 float64 — load a chunk slice
  (`zarr.open_array(...)[y0:y1, x0:x1]`) rather than `[:]` if you only need a region.
- **In-cluster / credentialed access:** every asset also carries an
  `alternate.s3.href` (`s3://esa-zarr-sentinel-explorer-fra/…`) for S3-native
  access with credentials, which additionally allows listing (and therefore plain
  `xr.open_zarr` on the group).